# US-1: Environment Setup Validation

This notebook validates that the Cloudflare Browser Rendering API credentials are correctly configured.

**Acceptance Criteria:**
- [x] AC1: All required Python packages installed
- [x] AC2: `.env` file exists with valid Cloudflare credentials
- [x] AC3: Credentials validated successfully via API test call
- [x] AC4: `.env` is listed in `.gitignore`
- [x] AC5: Python version is 3.8 or higher

## 1. Import Required Packages

In [2]:
import httpx
import polars as pl
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from tqdm import tqdm
import os
import sys

print(f"Python version: {sys.version}")
print(f"\nPackage versions:")
print(f"  httpx: {httpx.__version__}")
print(f"  polars: {pl.__version__}")
# print(f"  beautifulsoup4: {BeautifulSoup.__version__}")


Python version: 3.11.8 | packaged by conda-forge | (main, Feb 16 2024, 20:49:36) [Clang 16.0.6 ]

Package versions:
  httpx: 0.24.1
  polars: 1.35.2


## 2. Load Environment Variables

In [3]:
# Load .env file
load_dotenv()

# Get credentials
account_id = os.getenv("CLOUDFLARE_ACCOUNT_ID")
api_token = os.getenv("CLOUDFLARE_API_TOKEN")

# Validate credentials exist (show partial values for security)
if account_id and api_token:
    print(f"✅ Credentials loaded successfully")
    print(f"   Account ID: {account_id[:8]}...{account_id[-4:]}")
    print(f"   API Token: {api_token[:8]}...{api_token[-4:]}")
else:
    print("❌ Credentials not found in .env file")
    print("   Expected variables: CLOUDFLARE_ACCOUNT_ID, CLOUDFLARE_API_TOKEN")

✅ Credentials loaded successfully
   Account ID: 5decc977...c248
   API Token: cfat_HAO...be79


## 3. Test Cloudflare Crawl API

Submit a minimal crawl request to validate credentials work with the `/crawl` endpoint.

In [4]:
# Cloudflare Browser Rendering API endpoint
url = f"https://api.cloudflare.com/client/v4/accounts/{account_id}/browser-rendering/crawl"

headers = {
    "Authorization": f"Bearer {api_token}",
    "Content-Type": "application/json"
}

# Minimal test payload - crawl example.com with limit of 1 page
payload = {
    "url": "https://example.com",
    "limit": 1
}

print("Testing Cloudflare Browser Rendering CRAWL API...")
print(f"Endpoint: {url}")
print()

Testing Cloudflare Browser Rendering CRAWL API...
Endpoint: https://api.cloudflare.com/client/v4/accounts/5decc977ba3c9b36a3d2f089fc1ac248/browser-rendering/crawl



In [5]:
# Submit test crawl
with httpx.Client(timeout=60.0) as client:
    response = client.post(url, headers=headers, json=payload)
    result = response.json()

# Display results
print(f"API Response Status: {response.status_code}")
print(f"Success: {result.get('success')}")
print(f"Job ID: {result.get('result')}")
print()

API Response Status: 200
Success: True
Job ID: dc5bb15f-e63d-4c2e-b050-cc168498a9e0



In [6]:
# Validation summary
if response.status_code == 200 and result.get('success'):
    print("=" * 50)
    print("✅ CREDENTIALS VALIDATED SUCCESSFULLY!")
    print("=" * 50)
    print()
    print("The /crawl endpoint returned HTTP 200 with success=True.")
    print("US-1 Environment Setup is COMPLETE.")
    
    # Store job_id for cleanup
    job_id = result.get('result')
    print(f"\nTest Job ID: {job_id}")
    print("(This test job will auto-expire)")
else:
    print("❌ Credential validation FAILED")
    print(f"Status: {response.status_code}")
    print(f"Response: {result}")

✅ CREDENTIALS VALIDATED SUCCESSFULLY!

The /crawl endpoint returned HTTP 200 with success=True.
US-1 Environment Setup is COMPLETE.

Test Job ID: dc5bb15f-e63d-4c2e-b050-cc168498a9e0
(This test job will auto-expire)


## Summary

| AC | Criteria | Status |
|----|----------|--------|
| AC1 | Python packages installed | ✅ |
| AC2 | `.env` with valid credentials | ✅ |
| AC3 | Credentials validated via API | ✅ |
| AC4 | `.env` in `.gitignore` | ✅ |
| AC5 | Python 3.8+ | ✅ |

**Issue #1 can be closed.**